<a href="https://colab.research.google.com/github/khagenA/eng_labs_marry_me/blob/lite/marry_me.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementation Overview

## Architecture

This project implements an event-driven simulation using a **Producer → Queue → Worker Pool** design pattern.

- **Producer**: Reads events from a JSON file and injects them into the system according to their timestamps.
- **Team Queues**: Each team (Security, Catering, Waiters) has its own asynchronous queue.
- **Worker Pool**: Each team has multiple staff members that process events concurrently.

This structure models real-world dispatch systems where tasks are distributed across limited resources.

---

## Concurrency Model

The simulation uses **Python `asyncio`** to achieve non-blocking, asynchronous execution.

- Events arrive over time.
- Multiple workers can process incidents simultaneously.
- Workers operate independently without blocking the system.

This allows realistic time-based simulation within a single-threaded environment.

---

## Deadline & Event Classification

Each event has a priority-based deadline:

- **High** → 5 seconds  
- **Medium** → 10 seconds  
- **Low** → 15 seconds  

When processed, events are classified as:

- ✅ **Handled On Time** – Completed before deadline  
- ⚠️ **Delayed** – Completed after deadline  
- ❌ **Expired** – Started after deadline  
- ⏳ **Leftover** – Still waiting when simulation ends  

Stress increases for expired and delayed events.

---

## Simulation Lifecycle

- The simulation runs for **60 seconds**.
- Events are processed asynchronously.
- Workers continuously pull from their team queue.
- Final metrics summarize total received, handled, expired, delayed, leftover, and stress levels.

This implementation demonstrates event-driven design, deadline-aware scheduling, and concurrent worker management.


In [1]:
# ==========================================
# If running in Google Colab
# -----------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive"
PROJECT = "QWASAR/eng_labs_marry_me"

os.chdir(os.path.join(BASE_DIR, PROJECT))
print("Current directory:", os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content/drive/MyDrive/QWASAR/eng_labs_marry_me


In [2]:
import asyncio, json, time
from dataclasses import dataclass

WORK_SEC = 3.0
SIM_SEC  = 60.0

DEADLINE = {"high": 5.0, "medium": 10.0, "low": 15.0}

TYPE_TO_TEAM = {
    "brawl": "Security", "not_on_list": "Security",
    "bad_food": "Catering", "feeling_ill": "Catering",
    "dirty_table": "Waiters", "broken_item": "Waiters",
}

@dataclass
class Staff:
    status: str   # "Idle" | "Working"
    team: str

@dataclass(frozen=True)
class Event:
    id: int
    event_type: str
    priority: str
    description: str
    timestamp: float

def tnow(start): return time.monotonic() - start
def log(start, msg): print(f"[{tnow(start):6.2f}s] {msg}", flush=True)

async def worker_loop(staff: Staff, queue: asyncio.Queue, start, metrics):
    while True:
        ev = await queue.get()
        now = tnow(start)

        # if it's already past the deadline when we START, it truly expired
        deadline = ev.timestamp + DEADLINE[ev.priority]
        if now > deadline:
            metrics["expired"] += 1
            metrics["stress"] += 1
            log(start, f"EXPIRED  #{ev.id} team={staff.team}")
            queue.task_done()
            continue

        staff.status = "Working"
        log(start, f"START    #{ev.id} team={staff.team}")

        # simulate work
        await asyncio.sleep(WORK_SEC)

        staff.status = "Idle"
        finished = tnow(start)

        # classify: on-time vs late (delayed handled)
        if finished <= deadline:
            metrics["handled_on_time"] += 1
            log(start, f"ON-TIME  #{ev.id} team={staff.team}")
        else:
            metrics["delayed_handled"] += 1
            metrics["stress"] += 1  # late still stresses guests
            log(start, f"LATE     #{ev.id} team={staff.team} (deadline={deadline:.2f})")

        queue.task_done()

async def run_sim(path="events.json", workers_per_team=2):
    events = [Event(**e) for e in json.load(open(path))]
    events.sort(key=lambda e: e.timestamp)

    start = time.monotonic()
    metrics = {
        "received": 0,
        "handled_on_time": 0,
        "delayed_handled": 0,
        "expired": 0,
        "leftover": 0,
        "stress": 0,
        "invalid": 0,
    }

    queues = {team: asyncio.Queue() for team in ("Security", "Catering", "Waiters")}

    # start workers (multiple per team => true concurrency within team)
    for team in queues:
        for _ in range(workers_per_team):
            asyncio.create_task(worker_loop(Staff("Idle", team), queues[team], start, metrics))

    log(start, "Simulation started")

    # ingest events over time
    for ev in events:
        await asyncio.sleep(max(0.0, ev.timestamp - tnow(start)))

        metrics["received"] += 1
        team = TYPE_TO_TEAM.get(ev.event_type)
        if (team is None) or (ev.priority not in DEADLINE):
            metrics["invalid"] += 1
            log(start, f"INVALID  #{ev.id} type={ev.event_type} prio={ev.priority} -> SKIP")
            continue

        log(start, f"RECEIVED #{ev.id} -> {team}")
        await queues[team].put(ev)

       # run full sim duration
    await asyncio.sleep(max(0.0, SIM_SEC - tnow(start)))

    # wait until all already-queued events are fully processed (counted)
    await asyncio.gather(*(q.join() for q in queues.values()))
    await asyncio.sleep(0)

    # count leftovers still waiting (should usually be 0 after join)
    leftovers = 0
    for q in queues.values():
        while True:
            try:
                q.get_nowait()
            except asyncio.QueueEmpty:
                break
            else:
                leftovers += 1
                q.task_done()
    metrics["leftover"] = leftovers


    print("\nSUMMARY", flush=True)
    print("received         :", metrics["received"], flush=True)
    print("invalid          :", metrics["invalid"], flush=True)
    print("handled_on_time  :", metrics["handled_on_time"], flush=True)
    print("delayed_handled  :", metrics["delayed_handled"], flush=True)
    print("expired          :", metrics["expired"], flush=True)
    print("leftover         :", metrics["leftover"], flush=True)
    print("stress           :", metrics["stress"], flush=True)

    print("\nCHECK", flush=True)
    print("valid received =", metrics["received"] - metrics["invalid"], flush=True)
    print(
        "on_time + late + expired + leftover =",
        metrics["handled_on_time"] + metrics["delayed_handled"] + metrics["expired"] + metrics["leftover"],
        flush=True
    )


In [3]:
await run_sim("events_easy.json", workers_per_team=2)

[  0.00s] Simulation started
[  0.10s] RECEIVED #1 -> Waiters
[  0.10s] START    #1 team=Waiters
[  1.60s] RECEIVED #2 -> Security
[  1.60s] START    #2 team=Security
[  3.10s] ON-TIME  #1 team=Waiters
[  4.61s] ON-TIME  #2 team=Security
[  5.20s] RECEIVED #3 -> Security
[  5.20s] START    #3 team=Security
[  7.40s] RECEIVED #4 -> Catering
[  7.40s] START    #4 team=Catering
[  8.20s] ON-TIME  #3 team=Security
[ 10.41s] ON-TIME  #4 team=Catering
[ 14.00s] RECEIVED #5 -> Catering
[ 14.01s] START    #5 team=Catering
[ 17.01s] ON-TIME  #5 team=Catering
[ 23.80s] RECEIVED #6 -> Waiters
[ 23.80s] START    #6 team=Waiters
[ 24.50s] RECEIVED #7 -> Catering
[ 24.50s] START    #7 team=Catering
[ 24.70s] RECEIVED #8 -> Catering
[ 24.70s] START    #8 team=Catering
[ 26.81s] ON-TIME  #6 team=Waiters
[ 27.20s] RECEIVED #9 -> Catering
[ 27.50s] ON-TIME  #7 team=Catering
[ 27.51s] START    #9 team=Catering
[ 27.70s] ON-TIME  #8 team=Catering
[ 30.51s] ON-TIME  #9 team=Catering
[ 35.81s] RECEIVED #10 

In [4]:
await run_sim("events_medium.json", workers_per_team=2)


[  0.00s] Simulation started
[  0.40s] RECEIVED #1 -> Security
[  0.40s] START    #1 team=Security
[  1.40s] RECEIVED #2 -> Waiters
[  1.40s] START    #2 team=Waiters
[  1.50s] RECEIVED #3 -> Security
[  1.50s] START    #3 team=Security
[  1.60s] RECEIVED #4 -> Waiters
[  1.60s] START    #4 team=Waiters
[  3.40s] ON-TIME  #1 team=Security
[  4.41s] ON-TIME  #2 team=Waiters
[  4.50s] ON-TIME  #3 team=Security
[  4.60s] ON-TIME  #4 team=Waiters
[  4.80s] RECEIVED #5 -> Waiters
[  4.80s] START    #5 team=Waiters
[  5.10s] RECEIVED #6 -> Security
[  5.10s] START    #6 team=Security
[  5.30s] RECEIVED #7 -> Catering
[  5.30s] START    #7 team=Catering
[  7.81s] ON-TIME  #5 team=Waiters
[  8.10s] ON-TIME  #6 team=Security
[  8.30s] ON-TIME  #7 team=Catering
[  8.60s] RECEIVED #8 -> Security
[  8.60s] START    #8 team=Security
[ 10.90s] RECEIVED #9 -> Catering
[ 10.90s] START    #9 team=Catering
[ 11.60s] ON-TIME  #8 team=Security
[ 12.00s] RECEIVED #10 -> Security
[ 12.00s] START    #10 team

In [5]:
await run_sim("events_hard.json", workers_per_team=2)


[  0.00s] Simulation started
[  0.10s] RECEIVED #1 -> Security
[  0.10s] START    #1 team=Security
[  1.20s] RECEIVED #2 -> Security
[  1.20s] START    #2 team=Security
[  1.40s] RECEIVED #3 -> Security
[  1.60s] RECEIVED #4 -> Catering
[  1.60s] START    #4 team=Catering
[  1.60s] RECEIVED #5 -> Catering
[  1.60s] START    #5 team=Catering
[  1.70s] RECEIVED #6 -> Waiters
[  1.70s] START    #6 team=Waiters
[  1.90s] RECEIVED #7 -> Waiters
[  1.90s] START    #7 team=Waiters
[  3.11s] ON-TIME  #1 team=Security
[  3.11s] START    #3 team=Security
[  4.21s] ON-TIME  #2 team=Security
[  4.60s] ON-TIME  #4 team=Catering
[  4.60s] ON-TIME  #5 team=Catering
[  4.70s] ON-TIME  #6 team=Waiters
[  4.90s] ON-TIME  #7 team=Waiters
[  5.20s] RECEIVED #8 -> Security
[  5.20s] START    #8 team=Security
[  6.11s] ON-TIME  #3 team=Security
[  6.60s] RECEIVED #9 -> Security
[  6.60s] START    #9 team=Security
[  7.40s] RECEIVED #10 -> Catering
[  7.40s] START    #10 team=Catering
[  8.20s] ON-TIME  #8 t